# Example 12.6.5 / 1: Gaussian linear case

Companion notebook for **12.6.5 Chance Constraints and Risk** in the AMPL lab Documentation, section "Gaussian linear case". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Gaussian linear case

The fragment is not run here: `mean_return`, `covariance`, `x` and `target` belong to a
portfolio model, and the square root of a quadratic form needs a solver that accepts second-order
cones. The sign and quantile depend on whether the event is a lower or upper tail. Derive them from the event rather than copying a template.

The quantile $\Phi^{-1}(1-\epsilon)$ comes from `gsl_cdf_ugaussian_Pinv` in the GSL library. A
function from the library is declared with `function` in the model and works once the library is
loaded; `get_value` and `to_dict()` read the results. The example computes the capacity that a
load with mean 100 and standard deviation 10 needs:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
function gsl_cdf_ugaussian_Pinv;        # standard normal quantile, from the GSL library
set EPSILON;                            # allowed violation probabilities
param mean_load;                        # mean of the uncertain load a'x
param sd_load >= 0;                     # its standard deviation
param quantile {e in EPSILON} = gsl_cdf_ugaussian_Pinv(1 - e);
# Upper-tail event a'x <= b: b must cover the mean plus quantile standard deviations.
param required_capacity {e in EPSILON} = mean_load + quantile[e] * sd_load;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["EPSILON"] = [0.05, 0.01]
ampl.param["mean_load"] = 100
ampl.param["sd_load"] = 10

In [5]:
# run.py
ampl.eval("load amplgsl.dll;")  # amplpy has no method for loading a function library
print(ampl.param["quantile"].to_dict())
print(ampl.param["required_capacity"].to_dict())

{0.01: 2.3263478740408408, 0.05: 1.6448536269514715}
{0.01: 123.2634787404084, 0.05: 116.44853626951472}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set EPSILON := 0.05 0.01;
param mean_load := 100;
param sd_load := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplgsl.dll;
display quantile, required_capacity;

:     quantile required_capacity    :=
0.01   2.32635       123.263
0.05   1.64485       116.449
;

